# Goal and functionality of algorithm
Initiate organisms and watch them respond to stimuli through time. After a fixed number of time-steps, a reproduction condition triggers. Upon reproduction, the organisms which meet reproduction condition(s) reproduce (with a chance for mutation), all of the previous generation dies. This sequence repeats indefinitely. 


# Steps for a single generation
1. Initiate world (including state of n organisms) at t = 0
2. Allow organisms to perceive their situation and make a decision as to what to do. Taking action if applicable
3. Repeat step 2 for all timesteps

# Inter-generational steps
1. Run a single generation
2. Evaluate and execute reproduction condition. Reproduction should enable both passing of genetic information as well as the addition of new genetic information through mutation.
3. Repeat step 2 for n generations

# What does the MVP look like?
- organisms have a small number of neurons that map to some perception/action workflow
    - does this always need to look like [perception] -> [action]
- organisms can update state based on some perception of the world
- a population of organisms can reproduce based on some condition
- organisms can evolve (including passing genes and random mutations)

---
# Scratchpad

The MVP above is built. `uv run python execute.py` runs it with the animation; the cells below are for poking at individual creatures and working out *why* a behaviour appeared.

The answer to "does this always need to look like [perception] -> [action]" turned out to be no: inner neurons keep their value between timesteps, so a genome can wire perception -> memory -> action, or a loop that ignores perception entirely.

Start the kernel with `uv run jupyter lab` so the notebook picks up the project environment.

In [ ]:
%matplotlib inline
from dataclasses import replace

import matplotlib.pyplot as plt

from capability_utils import Action
from organism import CRITERIA, World
from settings import Settings

config = Settings()

## One organism, up close

In [ ]:
world = World(config=config)
org = world.organisms[0]

# Everything it can sense right now.
{str(sensor): round(value, 3) for sensor, value in org.perceive(world).items()}

In [ ]:
# Its whole brain, one connection per line.
print(org.brain.describe())
print()
print("senses it actually consults:", [str(s) for s in org.brain.needed_sensors])

In [ ]:
# What its action neurons want to do this timestep.
levels = org.brain.think(org, world)
{str(action): round(levels[action], 3) for action in Action}

## Watching a population evolve

In [ ]:
world = World(config=config, criterion=CRITERIA["left"])

history = [world.run_generation() / world.n_organisms for _ in range(40)]

plt.plot(history)
plt.xlabel("generation")
plt.ylabel("fraction surviving")
plt.ylim(0, 1);

In [ ]:
# Where an evolved generation ends up, versus where it started.
start_x, start_y = world.positions()
world.run_generation()
end_x, end_y = world.positions()

figure, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
for ax, (xs, ys), title in zip(
    axes,
    [(start_x, start_y), (end_x, end_y)],
    ["start of generation", "end of generation"],
    strict=True,
):
    ax.imshow(
        world.survival_zone_mask().T,
        origin="lower",
        cmap="Greens",
        alpha=0.25,
        extent=(0, world.width, 0, world.height),
    )
    ax.scatter(xs, ys, s=6)
    ax.set_title(title)
    ax.set_xlim(0, world.width)
    ax.set_ylim(0, world.height)

## Changing the rules without touching the code

`Settings` is frozen, so derive a variant with `replace` rather than editing globals.

In [ ]:
# How much does the mutation rate matter?
for rate in [0.005, 0.02, 0.08]:
    tuned = replace(config, point_mutation_rate=rate, n_organisms=150, steps_per_generation=100)
    trial = World(config=tuned, criterion=CRITERIA["corners"])
    curve = [trial.run_generation() / tuned.n_organisms for _ in range(25)]
    plt.plot(curve, label=f"mutation rate {rate}")

plt.xlabel("generation")
plt.ylabel("fraction surviving")
plt.ylim(0, 1)
plt.legend();

## Ideas to try

- A new survival criterion at the bottom of `organism.py` (the animation shades any zone automatically).
- A new member of `Sensor` or `Action` in `capability_utils.py` -- evolution picks it up on the next run with no other changes.
- Crossover between two survivors, instead of the current asexual mutation-only reproduction.